# 00 — ADLS Configuration & Data Ingestion
**Author:** Alvin David  
**Stack:** Azure Databricks · ADLS Gen2 · Azure Key Vault  

Authenticates Databricks to ADLS Gen2 using OAuth 2.0 via Service Principal.  
Downloads 12 months of NYC Yellow Taxi 2023 data (~607MB) directly from NYC TLC into ADLS raw container.

In [0]:
# Authentication 
# Credentials stored in Azure Key Vault, accessed via Databricks secret scope
# Never hardcoded — production security standard
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

RAW_PATH       = f"abfss://raw@{storage_account}.dfs.core.windows.net"
PROCESSED_PATH = f"abfss://processed@{storage_account}.dfs.core.windows.net"
CURATED_PATH   = f"abfss://curated@{storage_account}.dfs.core.windows.net"

print("ADLS configured successfully")
print(f"RAW       → {RAW_PATH}")
print(f"PROCESSED → {PROCESSED_PATH}")
print(f"CURATED   → {CURATED_PATH}")

In [0]:
# ── Connection Verification ─────────────────────────────────
# ls() proves OAuth token was issued and ADLS is accessible
dbutils.fs.ls(f"{RAW_PATH}/")
print("Connected to ADLS successfully")

In [0]:
# ── Ingest NYC Yellow Taxi 2023 Data ───────────────────────
# Downloads 12 monthly Parquet files directly from NYC TLC CloudFront
# Each file ~50MB, total ~607MB — stored in raw/yellow_taxi/
# Skip logic prevents re-downloading on subsequent runs
import urllib.request
import os


base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data"
months = [f"{m:02d}" for m in range(1, 13)]

for month in months:
    filename = f"yellow_tripdata_2023-{month}.parquet"
    url = f"{base_url}/{filename}"
    local_path = f"/tmp/{filename}"
    adls_path = f"{RAW_PATH}/yellow_taxi/{filename}"
    
    # Check if already exists
    try:
        dbutils.fs.ls(adls_path)
        print(f"Already exists: {filename}")
        continue
    except:
        pass
    
    # Download to driver temp storage
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, local_path)
    
    # Copy to ADLS
    dbutils.fs.cp(f"file://{local_path}", adls_path)
    os.remove(local_path)
    print(f"Uploaded: {filename}")

print("All files loaded to ADLS")

In [0]:
# ── Dimension Table — Taxi Zone Lookup ─────────────────────
# 265-row CSV mapping LocationID to Borough and Zone name
# Used in Silver layer for broadcast join (avoids 35M row shuffle)
import urllib.request

url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
local_path = "/tmp/taxi_zone_lookup.csv"
adls_path = f"{RAW_PATH}/lookup/taxi_zone_lookup.csv"

urllib.request.urlretrieve(url, local_path)
dbutils.fs.cp(f"file://{local_path}", adls_path)

print("Zone lookup uploaded")

# Preview it
df = spark.read.option("header", True).csv(adls_path)
df.show(5)